In [48]:
from pathlib import Path
from typing import Tuple, Optional, Iterable, Callable, List, Dict
from collections import OrderedDict
from copy import deepcopy

import cv2
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
from sklearn.model_selection import train_test_split

import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    swin_s,
    Swin_S_Weights,
    efficientnet_b7,
    EfficientNet_B7_Weights,
    efficientnet_b3,
    EfficientNet_B3_Weights,
    swin_v2_t,
    Swin_V2_T_Weights,
    swin_v2_s,
    Swin_V2_S_Weights,
    vit_b_16,
    ViT_B_16_Weights,
    densenet201,
    DenseNet201_Weights,
    googlenet,
    GoogLeNet_Weights,
    convnext_small,
    ConvNeXt_Small_Weights,
    regnet_y_8gf,
    RegNet_Y_8GF_Weights,
)
from torchvision.models.swin_transformer import Permute
from torchvision.io import read_image
from torchvision import transforms
from torchvision.ops import FeaturePyramidNetwork

from tqdm import tqdm

In [49]:
data_root = Path("data/dataset_crop")

df = pd.read_csv(data_root / "thread_depths.csv")
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)

In [50]:
import os
class ThreadDataset(Dataset):
    def __init__(self, data: pd.DataFrame, data_root_dir: str, transform: Optional[nn.Module] = None):
        self.data = data
        self.data_root_dir = Path(data_root_dir)
        self.image_paths = []
        self.labels = []
        for _, row in self.data.iterrows():
            image_path = self.data_root_dir / row["path"]
            if not image_path.exists():
                print(f"Warning: {image_path} does not exist")
                continue
            self.image_paths.append(self.data_root_dir / row["path"])
            self.labels.append(row["label"])

        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, float]:
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = read_image(str(image_path))
        if self.transform is not None:
            image = self.transform(image)

        name = os.path.basename(image_path)
        return image, label, name

In [51]:
transform = transforms.Compose(
    [
        # Clahe(),
        lambda x: x / 255,
        transforms.Resize(
            (512, 512), interpolation=transforms.InterpolationMode.BICUBIC
        ),
    ]
)

transform_aug = transforms.Compose(
    [
        transform,
        # transforms.ToPILImage(),
        # transforms.RandAugment(
        #     num_ops=3, interpolation=transforms.InterpolationMode.BICUBIC
        # ),
        # transforms.ToTensor(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        # transforms.RandomAffine(15, (0.05, 0.05), fill=255),
    ]
)

In [52]:
train_dataset = ThreadDataset(df_train, data_root, transform_aug)
val_dataset = ThreadDataset(df_val, data_root, transform)

train_loader = DataLoader(train_dataset, shuffle=True, num_workers=8, batch_size=8)
val_loader = DataLoader(val_dataset, shuffle=False, num_workers=8, batch_size=16)

/home/nikkimen/miniconda3/envs/kolobok/lib/python3.11/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [53]:
models = {}

models["swin_v2"] = swin_v2_t(weights=Swin_V2_T_Weights.IMAGENET1K_V1)
models["swin_v2"].head = nn.Sequential(nn.Linear(768, 512), nn.GELU(), nn.Linear(512, 1))

models["effnet_b7"] = efficientnet_b7(weights=EfficientNet_B7_Weights.IMAGENET1K_V1)
models["effnet_b7"].classifier = nn.Sequential(nn.Linear(2560, 512), nn.SiLU(), nn.Linear(512, 1))

models["densenet201"] = densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)
models["densenet201"].classifier = nn.Sequential(nn.Linear(1920, 512), nn.ReLU(), nn.Linear(512, 1))

models["googlenet"] = googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1)
models["googlenet"].fc = nn.Sequential(nn.Linear(1024, 512), nn.ReLU(), nn.Linear(512, 1))

In [54]:
%cd checkpoints/

/home/nikkimen/Documents/Work/kolobok/checkpoints


In [55]:
models["effnet_b7"].load_state_dict(torch.load('effnet_b7_0.8172.pt', map_location='cpu'))
models["densenet201"].load_state_dict(torch.load('densenet201_0.8172.pt', map_location='cpu'))
models["googlenet"].load_state_dict(torch.load('googlenet_0.8138.pt', map_location='cpu'))
models["swin_v2"].load_state_dict(torch.load('swin_v2_0.8034.pt', map_location='cpu'))

<All keys matched successfully>

In [58]:
%cd ..

/home/nikkimen/Documents/Work/kolobok


In [59]:
results = []

for model_name, model in models.items():
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    with torch.no_grad():
        for images, labels, names in tqdm(
            train_loader, 
            total=len(train_loader)
        ):

            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images).exp()

            for label, name, out in zip(labels, names, outputs):
                results.append({
                    'weight_file': model_name,
                    'image_name': name,
                    'image_label' : label,
                    'output'     : float(out)
                })

df = pd.DataFrame(results)
df

  0%|          | 0/145 [00:00<?, ?it/s]/home/nikkimen/miniconda3/envs/kolobok/lib/python3.11/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


100%|██████████| 145/145 [07:57<00:00,  3.29s/it]


,weight_file,image_name,image_label,output
0,swin_v2,1245.png,"[tensor(8.2000, dtype=torch.float64)]",8.805075
1,swin_v2,22.png,"[tensor(2.2000, dtype=torch.float64)]",2.082259
2,swin_v2,1097.png,"[tensor(9., dtype=torch.float64)]",9.452421
3,swin_v2,517.png,"[tensor(2., dtype=torch.float64)]",2.027646
4,swin_v2,742.png,"[tensor(3.2000, dtype=torch.float64)]",2.873113
...,...,...,...,...
4623,googlenet,1427.png,"[tensor(4., dtype=torch.float64)]",4.340942
4624,googlenet,252.png,"[tensor(7., dtype=torch.float64)]",7.064976
4625,googlenet,547.png,"[tensor(4.7000, dtype=torch.float64)]",4.283363
4626,googlenet,1034.png,"[tensor(5.8000, dtype=torch.float64)]",5.720842


In [60]:
df.to_csv(
    'data/stacking_train.csv',   # output file
    index=False,           # don’t write pandas’ row numbers
    header=True,           # write column names
    sep=',',               # delimiter
    encoding='utf-8'       # encoding
)

In [61]:
results = []

for model_name, model in models.items():
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    with torch.no_grad():
        for images, labels, names in tqdm(
            val_loader, 
            total=len(val_loader)
        ):

            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images).exp()

            for label, name, out in zip(labels, names, outputs):
                results.append({
                    'weight_file': model_name,
                    'image_name': name,
                    'image_label' : label,
                    'output'     : float(out)
                })

df = pd.DataFrame(results)
df

  0%|          | 0/19 [00:00<?, ?it/s]/home/nikkimen/miniconda3/envs/kolobok/lib/python3.11/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
100%|██████████| 19/19 [01:56<00:00,  6.14s/it]


,weight_file,image_name,image_label,output
0,swin_v2,1023.png,"[tensor(4.1000, dtype=torch.float64)]",3.994047
1,swin_v2,381.png,"[tensor(4.3000, dtype=torch.float64)]",4.128712
2,swin_v2,843.png,"[tensor(6., dtype=torch.float64)]",6.015761
3,swin_v2,427.png,"[tensor(6.3000, dtype=torch.float64)]",5.380384
4,swin_v2,192.png,"[tensor(4.8000, dtype=torch.float64)]",3.896557
...,...,...,...,...
1155,googlenet,796.png,"[tensor(6., dtype=torch.float64)]",6.237909
1156,googlenet,571.png,"[tensor(3.4000, dtype=torch.float64)]",3.285386
1157,googlenet,86.png,"[tensor(6.7000, dtype=torch.float64)]",5.895592
1158,googlenet,1374.png,"[tensor(3.6000, dtype=torch.float64)]",3.785126


In [62]:
df.to_csv(
    'data/stacking_val.csv',   # output file
    index=False,           # don’t write pandas’ row numbers
    header=True,           # write column names
    sep=',',               # delimiter
    encoding='utf-8'       # encoding
)

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error  # or accuracy_score for classification

# 1) Read in your CSV
df = pd.read_csv("stacking_train.csv")

def parse_label(x):
    # x might be something like "tensor([6.4000], dtype=torch.float64)"
    # or a tuple string "(tensor([6.4000], dtype=...),4)"
    # We'll just grab the last floating‐point number in the string.
    nums = re.findall(r"[\d\.]+", str(x))
    # assume the true label is the last number
    return float(nums[0])

df["image_label"] = df["image_label"].apply(parse_label)

# 2) Pivot so each model’s prediction becomes its own column
wide = df.pivot(
    index="image_name",        # one row per image
    columns="weight_file",     # one column per base model
    values="output"            # the model’s numeric output
).reset_index()

# 3) Bring the ground-truth label back in
#    (assuming you also stored `image_label` in your long DataFrame)
labels = (
    df[["image_name", "image_label"]]
    .drop_duplicates(subset="image_name")
    .set_index("image_name")
)
wide = wide.set_index("image_name").join(labels).reset_index()

# 4) Prepare X, y
X = wide.drop(["image_name", "image_label"], axis=1)
y = wide["image_label"]

# 6) Fit an XGBoost meta-model
#    (choose XGBRegressor if it’s a regression task; XGBClassifier if classification)
meta_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
meta_model.fit(X, y)

# # 7) Evaluate
# y_pred = meta_model.predict(X_val)
# rmse = mean_squared_error(y_val, y_pred, squared=False)
# print(f"Validation RMSE: {rmse:.4f}")

# # 8) (Optional) Retrain on the full dataset
# meta_model.fit(X, y)

# # 9) Save your trained meta-model for later
# import joblib
# joblib.dump(meta_model, "xgb_stacker.joblib")


In [ ]:
import numpy as np
df_val = pd.read_csv("stacking_val.csv")

def parse_label(x):
    # x might be something like "tensor([6.4000], dtype=torch.float64)"
    # or a tuple string "(tensor([6.4000], dtype=...),4)"
    # We'll just grab the last floating‐point number in the string.
    nums = re.findall(r"[\d\.]+", str(x))
    # assume the true label is the last number
    return float(nums[0])

df_val["image_label"] = df_val["image_label"].apply(parse_label)

# 2) Pivot so each model’s prediction becomes its own column
wide = df_val.pivot(
    index="image_name",        # one row per image
    columns="weight_file",     # one column per base model
    values="output"            # the model’s numeric output
).reset_index()

# 3) Bring the ground-truth label back in
#    (assuming you also stored `image_label` in your long DataFrame)
labels = (
    df_val[["image_name", "image_label"]]
    .drop_duplicates(subset="image_name")
    .set_index("image_name")
)
wide = wide.set_index("image_name").join(labels).reset_index()

# 4) Prepare X, y
X_val = wide.drop(["image_name", "image_label"], axis=1)
y_val = wide["image_label"]

y_pred = meta_model.predict(X_val)

errors = abs(y_val - y_pred)
fraction_within_1 = np.mean(errors <= 1)
print(f"Fraction of predictions with |error| ≤ 1: {fraction_within_1:.3f}")